# 03 — Optimization

Channel selection (CSP ranking + genetic algorithm) and data augmentation
to squeeze maximum performance from the best pipeline.

- CSP-based channel importance ranking
- GA-optimized channel subset selection
- Data augmentation: Gaussian noise, FT surrogates, hemisphere recombination

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import mne

from src.loading import load_all_patients, CH_NAMES, LEFT_IDX, RIGHT_IDX
from src.channel_selection import csp_rank_channels, ga_channel_selection
from src.preprocessing import (
    augment_gaussian_noise,
    augment_ft_surrogate,
    augment_hemisphere_recombination,
)
from src.classifiers import build_all_pipelines, evaluate_all
from src.evaluation import compare_to_baseline

mne.set_log_level('WARNING')

## 3.1 Load Data

In [ ]:
patient_data = load_all_patients('../data/')

## 3.2 CSP Channel Ranking

In [ ]:
for pid, (X, y, _) in patient_data.items():
    print(f'\n--- {pid} ---')
    ranked_idx, scores = csp_rank_channels(X, y)
    for i, (idx, score) in enumerate(zip(ranked_idx[:8], scores[:8])):
        print(f'  #{i+1}: {CH_NAMES[idx]:>4s} (importance: {score:.4f})')

## 3.3 Genetic Algorithm Channel Selection

Evolves binary channel masks to maximize TS+LR cross-validation accuracy.

In [ ]:
ga_results = {}

for pid, (X, y, _) in patient_data.items():
    print(f'\nGA for {pid}...')
    result = ga_channel_selection(X, y, n_gen=30, pop_size=50)
    ga_results[pid] = result
    selected = [CH_NAMES[i] for i in result['best_channels']]
    print(f'  Best accuracy: {result["best_accuracy"]:.3f}')
    print(f'  Selected channels ({len(selected)}): {selected}')

## 3.4 Data Augmentation Comparison

Test different augmentation strategies on the best Riemannian pipeline.

In [ ]:
from src.classifiers import build_all_pipelines

# Use TS+LR as the benchmark pipeline
pipe = build_all_pipelines()['TS+LR']

for pid, (X, y, _) in patient_data.items():
    print(f'\n--- {pid} ---')
    
    # Baseline (no augmentation)
    from sklearn.model_selection import cross_val_score, StratifiedKFold
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores_base = cross_val_score(pipe, X, y, cv=cv)
    print(f'  Baseline:     {scores_base.mean():.3f} ± {scores_base.std():.3f}')
    
    # Gaussian noise
    X_gn, y_gn = augment_gaussian_noise(X, y, std=0.1, n_copies=2)
    scores_gn = cross_val_score(pipe, X_gn, y_gn, cv=cv)
    print(f'  +Gaussian:    {scores_gn.mean():.3f} ± {scores_gn.std():.3f}')
    
    # FT surrogates
    X_ft, y_ft = augment_ft_surrogate(X, y, phase_noise_mag=0.3, n_copies=2)
    scores_ft = cross_val_score(pipe, X_ft, y_ft, cv=cv)
    print(f'  +FT surrogate: {scores_ft.mean():.3f} ± {scores_ft.std():.3f}')
    
    # Hemisphere recombination
    X_hr, y_hr = augment_hemisphere_recombination(X, y, LEFT_IDX, RIGHT_IDX, n_copies=2)
    scores_hr = cross_val_score(pipe, X_hr, y_hr, cv=cv)
    print(f'  +Hemisphere:  {scores_hr.mean():.3f} ± {scores_hr.std():.3f}')